# Step 10b-X: sinh query mở rộng cho 4 bộ mới, có lưu logprobs (GPU)

Cùng thiết lập với Step 1C: LLaMA 3.1-8B-Instruct, vLLM fp16, TP=2, greedy, 256 token, prompt Phụ lục G. Điểm mới duy nhất là `--logprobs 5`, dùng sau này cho phân tích độ tự tin; văn bản sinh ra không đổi.

- **Accelerator:** GPU T4 ×2. **Internet:** On. **Secret:** `HF_TOKEN` (LLaMA là mô hình gated, như Step 1C).
- **Input:** `qudar-beir-data-2`, `qudar-patches` (bản mới).
- **Thời gian ước tính:** 21.088 query ở khoảng 5–6 query/giây → khoảng 65–75 phút, cộng 3 lần nạp mô hình × 3–4 phút. Tổng khoảng 1,5 giờ.
- **Sau khi chạy:** tạo dataset `qudar-expansions-2` từ thư mục `expansions/` (gồm file JSON và thư mục `logprobs/`).
- Bị ngắt giữa chừng: chạy lại các ô, phần đã sinh sẽ được bỏ qua.

In [1]:
import glob, os
DATA2 = os.path.dirname(os.path.dirname(glob.glob("/kaggle/input/**/qudar-beir-data-2/**/dbpedia-entity/corpus.jsonl", recursive=True)[0]))
from kaggle_secrets import UserSecretsClient
os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")
PATCH = os.path.dirname(glob.glob("/kaggle/input/**/expand_queries.py", recursive=True)[0])
assert "--logprobs" in open(f"{PATCH}/expand_queries.py").read(), "qudar-patches chưa phải bản mới"
!pip install -q vllm
!pip install -q --no-deps beir
E = f"python {PATCH}/expand_queries.py --data-root {DATA2} --out-dir /kaggle/working/expansions --logprobs 5"

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
ydata-profiling 4.18.4 requires numba<0.63,>=0.60, but you have numba 0.65.0 which is incompatible.
litellm 1.109.1 requires huggingface-hub<2.0,>=0.34.0, but you have huggingface-hub 2.4.0 which is incompatible.
numba-cuda 0.28.2 requires cuda-core<1.0.0,>=0.5.1, but you have cuda-core 1.3.1 which is incompatible.
datasets 4.8.5 requires huggingface-hub<2.0,>=0.25.0, but you have huggingface-hub 2.4.0 which is incompatible.
cuml-cu13 26.6.0 requires numba<0.65.0,>=0.60.0, but you have numba 0.65.0 which is incompatible.
cudf-cu13 26.6.0 requires numba<0.65.0,>=0.60.0, but you have numba 0.65.0 which is incompatible.
gradio 6.30.0 requires pillow<13.0,>=8.0, but you have pillow 13.0.0 which is incompatible.
google-adk 2.7.1 requires opentelemetry-api<=1.42.1,>=1.39, but you have opentelemetry-api 1.45.2 which is i

In [2]:
# Test của 4 bộ: 300 + 323 + 6666 + 400 = 7689 query
!{E} --datasets scifact nfcorpus fever dbpedia-entity --split test

scifact/test: 300 queries, 0 done, 300 to generate
W1123 08:11:30.803000 80 torch/_opaque_base.py:6] torch._opaque_base is deprecated, use torch._custom_class_base instead
W1123 08:11:30.804000 80 torch/_library/opaque_object.py:288] register_opaque_type is deprecated, use register_custom_class instead
W1123 08:11:30.804000 80 torch/_library/opaque_object.py:211] typ='value' is deprecated, use typ='constant' instead
(MainProcess pid=80) INFO 11-23 08:11:54 [startup.py:96] non-default args: {'dtype': 'half', 'max_model_len': 2048, 'tensor_parallel_size': 2, 'gpu_memory_utilization': 0.9, 'disable_log_stats': True, 'model': 'meta-llama/Llama-3.1-8B-Instruct'}
config.json: 100%|█████████████████████████████| 855/855 [00:00<00:00, 4.52MB/s]
(MainProcess pid=80) INFO 11-23 08:12:11 [model.py:761] Resolved architecture: LlamaForCausalLM
(MainProcess pid=80) WARNING 11-23 08:12:11 [model.py:2560] Casting torch.bfloat16 to torch.float16.
(MainProcess pid=80) INFO 11-23 08:12:11 [model.py:2214]

In [3]:
# Train: SciFact 809 + NFCorpus 2590
!{E} --datasets scifact nfcorpus --split train

scifact/train: 809 queries, 0 done, 809 to generate
W1123 08:40:10.102000 501 torch/_opaque_base.py:6] torch._opaque_base is deprecated, use torch._custom_class_base instead
W1123 08:40:10.103000 501 torch/_library/opaque_object.py:288] register_opaque_type is deprecated, use register_custom_class instead
W1123 08:40:10.103000 501 torch/_library/opaque_object.py:211] typ='value' is deprecated, use typ='constant' instead
(MainProcess pid=501) INFO 11-23 08:40:24 [startup.py:96] non-default args: {'dtype': 'half', 'max_model_len': 2048, 'tensor_parallel_size': 2, 'gpu_memory_utilization': 0.9, 'disable_log_stats': True, 'model': 'meta-llama/Llama-3.1-8B-Instruct'}
(MainProcess pid=501) INFO 11-23 08:40:26 [model.py:761] Resolved architecture: LlamaForCausalLM
(MainProcess pid=501) WARNING 11-23 08:40:26 [model.py:2560] Casting torch.bfloat16 to torch.float16.
(MainProcess pid=501) INFO 11-23 08:40:26 [model.py:2214] Using max model len 2048
(MainProcess pid=501) INFO 11-23 08:40:26 [sche

In [4]:
# Train FEVER: 10.000 query đầu (như HotpotQA)
!{E} --datasets fever --split train --max-queries 10000

fever/train: 10000 queries, 0 done, 10000 to generate
W1123 08:53:14.603000 711 torch/_opaque_base.py:6] torch._opaque_base is deprecated, use torch._custom_class_base instead
W1123 08:53:14.604000 711 torch/_library/opaque_object.py:288] register_opaque_type is deprecated, use register_custom_class instead
W1123 08:53:14.604000 711 torch/_library/opaque_object.py:211] typ='value' is deprecated, use typ='constant' instead
(MainProcess pid=711) INFO 11-23 08:53:31 [startup.py:96] non-default args: {'dtype': 'half', 'max_model_len': 2048, 'tensor_parallel_size': 2, 'gpu_memory_utilization': 0.9, 'disable_log_stats': True, 'model': 'meta-llama/Llama-3.1-8B-Instruct'}
(MainProcess pid=711) INFO 11-23 08:53:32 [model.py:761] Resolved architecture: LlamaForCausalLM
(MainProcess pid=711) WARNING 11-23 08:53:32 [model.py:2560] Casting torch.bfloat16 to torch.float16.
(MainProcess pid=711) INFO 11-23 08:53:32 [model.py:2214] Using max model len 2048
(MainProcess pid=711) INFO 11-23 08:53:32 [sc

In [5]:
# Tóm tắt để gửi lại
import json, numpy as np
from pathlib import Path
base = Path("/kaggle/working/expansions")
expect = {"scifact_test": 300, "scifact_train": 809, "nfcorpus_test": 323, "nfcorpus_train": 2590,
          "fever_test": 6666, "fever_train": 10000, "dbpedia-entity_test": 400}
for key, n in expect.items():
    ds, split = key.rsplit("_", 1)
    exp = json.loads((base / f"expanded_queries_{ds}_{split}_expansion.json").read_text())
    words = [len(v.split()) for v in exp.values()]
    parts = sorted((base / "logprobs" / key).glob("part_*.npz"))
    lp_q = set(); toks = 0; mean_lp = []
    for p in parts:
        z = np.load(p)
        lp_q.update(z["qids"].tolist()); toks += len(z["token_ids"]); mean_lp.append(float(np.nanmean(z["chosen_lp"])))
    print(f"{key}: {len(exp)}/{n} expansion, {sum(w == 0 for w in words)} rỗng, {np.mean(words):.0f} từ TB | "
          f"logprobs: {len(lp_q)} query, {toks} token, log-prob TB {np.mean(mean_lp):.3f}")
!du -sh /kaggle/working/expansions

scifact_test: 300/300 expansion, 0 rỗng, 144 từ TB | logprobs: 300 query, 56082 token, log-prob TB -0.388
scifact_train: 809/809 expansion, 0 rỗng, 145 từ TB | logprobs: 809 query, 150878 token, log-prob TB -0.395
nfcorpus_test: 323/323 expansion, 0 rỗng, 143 từ TB | logprobs: 323 query, 57816 token, log-prob TB -0.397
nfcorpus_train: 2590/2590 expansion, 0 rỗng, 141 từ TB | logprobs: 2590 query, 459307 token, log-prob TB -0.398
fever_test: 6666/6666 expansion, 0 rỗng, 126 từ TB | logprobs: 6666 query, 1070269 token, log-prob TB -0.433
fever_train: 10000/10000 expansion, 0 rỗng, 127 từ TB | logprobs: 10000 query, 1639485 token, log-prob TB -0.411
dbpedia-entity_test: 400/400 expansion, 0 rỗng, 143 từ TB | logprobs: 400 query, 73974 token, log-prob TB -0.407
122M	/kaggle/working/expansions
